In [1]:
#Imports y constantes

import pandas as pd
import numpy as np
import glob
from scipy.optimize import minimize
from scipy.stats import poisson
from tqdm import tqdm

BASE = "C:/Users/Usuario/historical_football_data"
MAX_GOALS = 10
PUNTOS_EXACTO = 6
PUNTOS_RESULTADO = 3
N_PARTIDOS = 104  # partidos del Mundial

In [2]:
archivos = [f for f in glob.glob(f"{BASE}/*.csv") if "combined" not in f]

dfs = []
for path in archivos:
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data["Date"] = pd.to_datetime(data["Date"], dayfirst=True)

cols = ["Date", "Div", "HomeTeam", "AwayTeam", "FTHG", "FTAG",
        "PSCH", "PSCD", "PSCA",
        "PC>2.5", "PC<2.5",
        "PCAHH", "PCAHA", "AHCh"]

data = data[cols].dropna().copy()
data = data.rename(columns={"PC>2.5": "PC_over", "PC<2.5": "PC_under"})
data["FTHG"] = data["FTHG"].astype(int)
data["FTAG"] = data["FTAG"].astype(int)

print(f"Partidos con datos completos: {len(data)}")
print(data["Date"].dt.year.value_counts().sort_index())
print(data["Div"].value_counts())

# Detectar filas con cuotas inválidas
invalidas = data[
    (data["PC_over"] <= 1) | (data["PC_under"] <= 1) |
    (data["PSCH"] <= 1) | (data["PSCD"] <= 1) | (data["PSCA"] <= 1)
]
print(f"Partidos con cuotas inválidas: {len(invalidas)}")
print(invalidas[["Date", "Div", "HomeTeam", "AwayTeam", 
                  "PSCH", "PSCD", "PSCA", "PC_over", "PC_under"]].to_string())

# Eliminar cuotas inválidas (0 o menores a 1)
cols_odds = ["PSCH", "PSCD", "PSCA", "PC_over", "PC_under", "PCAHH", "PCAHA"]
data = data[(data[cols_odds] > 1).all(axis=1)]
print(f"Partidos tras eliminar cuotas inválidas: {len(data)}")

Partidos con datos completos: 7991
Date
2021     894
2022    1677
2023    1908
2024    1695
2025    1722
2026      95
Name: count, dtype: int64
Div
E0     1719
I1     1718
SP1    1706
F1     1520
D1     1328
Name: count, dtype: int64
Partidos con cuotas inválidas: 3
           Date  Div       HomeTeam    AwayTeam  PSCH  PSCD   PSCA  PC_over  PC_under
3227 2025-04-19   F1       Paris SG    Le Havre  1.16  9.25  14.64      0.0       0.0
4788 2025-05-10   D1  Bayern Munich  M'gladbach  1.14  9.50  16.50      0.0       0.0
8609 2025-10-18  SP1      Barcelona      Girona  1.21  7.51  12.72      0.0       0.0
Partidos tras eliminar cuotas inválidas: 7988


In [3]:
def remove_margin(probs):
    total = sum(probs)
    return [p / total for p in probs]

def fair_1x2(odds_h, odds_d, odds_a):
    f = remove_margin([1/odds_h, 1/odds_d, 1/odds_a])
    return {"home": f[0], "draw": f[1], "away": f[2]}

def fair_ou(odds_over, odds_under):
    f = remove_margin([1/odds_over, 1/odds_under])
    return {"over": f[0], "under": f[1]}

# Vectorizado — ~50x más rápido que el doble loop
def build_matrix(lam_h, lam_a):
    i = np.arange(MAX_GOALS + 1)
    return np.outer(poisson.pmf(i, lam_h), poisson.pmf(i, lam_a))

def matrix_1x2(matrix):
    return (np.sum(np.tril(matrix, -1)),
            np.sum(np.diag(matrix)),
            np.sum(np.triu(matrix, 1)))

def matrix_ou(matrix, line=2.5):
    i = np.arange(MAX_GOALS + 1)
    # Máscara vectorizada
    mask = (i[:, None] + i[None, :]) > line
    p_over = np.sum(matrix[mask])
    return p_over, 1 - p_over

def apply_dixon_coles(matrix, lam_h, lam_a, rho=-0.13):
    m = matrix.copy()
    m[0,0] *= 1 - lam_h * lam_a * rho
    m[1,0] *= 1 + lam_a * rho
    m[0,1] *= 1 + lam_h * rho
    m[1,1] *= 1 - rho
    return m / m.sum()

def infer_lambdas(f1x2, fou, ah_line=None):
    def loss(params):
        lh, la = params
        if lh <= 0 or la <= 0:
            return 1e6
        mat = build_matrix(lh, la)
        ph, pd_, pa = matrix_1x2(mat)
        po, pu = matrix_ou(mat)
        err = ((ph - f1x2["home"])**2 +
               (pd_ - f1x2["draw"])**2 +
               (pa - f1x2["away"])**2 +
               (po - fou["over"])**2 +
               (pu - fou["under"])**2)
        if ah_line is not None:
            # Restricción sobre P(AH) en vez de diferencia de lambdas
            i = np.arange(MAX_GOALS + 1)
            diff = i[:, None] - i[None, :]
            mask = diff + ah_line > 0
            p_ah = float(np.sum(mat[mask]))
            err += 0.3 * (p_ah - 0.5)**2
        return err

    total = max(1.5, min(2.5 + (fou["over"] - 0.5) * 2, 6.0))
    share = f1x2["home"] / (f1x2["home"] + f1x2["away"] + 1e-9)
    lh0 = total * (0.5 + 0.3 * (share - 0.5))
    la0 = total - lh0

    res = minimize(loss, x0=[lh0, la0], method="Nelder-Mead",
                   options={"xatol": 1e-5, "fatol": 1e-6, "maxiter": 2000})
    return max(res.x[0], 0.1), max(res.x[1], 0.1)

def get_result(h, a):
    if h > a: return "home"
    if h == a: return "draw"
    return "away"

def score_points(pred_h, pred_a, real_h, real_a):
    if pred_h == real_h and pred_a == real_a:
        return PUNTOS_EXACTO
    if get_result(pred_h, pred_a) == get_result(real_h, real_a):
        return PUNTOS_RESULTADO
    return 0

def best_score_max_exact(matrix):
    idx = np.unravel_index(np.argmax(matrix), matrix.shape)
    return int(idx[0]), int(idx[1])

# Vectorizado — en vez de doble loop sobre 121 celdas
def best_score_max_ev(matrix, pts_exacto=None, pts_resultado=None):
    pe = pts_exacto if pts_exacto is not None else PUNTOS_EXACTO
    pr = pts_resultado if pts_resultado is not None else PUNTOS_RESULTADO
    
    i = np.arange(MAX_GOALS + 1)
    result_matrix = np.where(i[:, None] > i[None, :], 0,
                    np.where(i[:, None] == i[None, :], 1, 2))

    ev_matrix = np.zeros((MAX_GOALS+1, MAX_GOALS+1))
    for res_val in [0, 1, 2]:
        mask = result_matrix == res_val
        p_result_total = np.sum(matrix[mask])
        for ii in range(MAX_GOALS+1):
            for jj in range(MAX_GOALS+1):
                if result_matrix[ii, jj] == res_val:
                    p_exact = matrix[ii, jj]
                    p_result = p_result_total - p_exact
                    ev_matrix[ii, jj] = (p_exact * pe + p_result * pr)

    idx = np.unravel_index(np.argmax(ev_matrix), ev_matrix.shape)
    return int(idx[0]), int(idx[1])

    
def best_score_max_ev_with_draws(matrix, p_draw_market=None, draw_threshold=0.28):
    p_draw = p_draw_market if p_draw_market is not None else sum(matrix[k,k] for k in range(MAX_GOALS+1))
    
    if p_draw > draw_threshold:
        best_ev = -1
        best = (1, 1)
        for k in range(MAX_GOALS + 1):
            p_exact = matrix[k, k]
            p_other_draws = sum(matrix[i,i] for i in range(MAX_GOALS+1) if i != k)
            ev = p_exact * PUNTOS_EXACTO + p_other_draws * PUNTOS_RESULTADO
            if ev > best_ev:
                best_ev = ev
                best = (k, k)
        return best
    else:
        return best_score_max_ev(matrix)

In [35]:
# Verificar distribución de P(draw) en la matriz vs mercado
p_draws_matrix = []
p_draws_market = []

for _, row in data.sample(500, random_state=42).iterrows():
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = fair_ou(row["PC_over"], row["PC_under"])
    lh, la = infer_lambdas(f, fou)
    matrix = build_matrix(lh, la)
    
    p_draw_matrix = sum(matrix[k, k] for k in range(MAX_GOALS + 1))
    p_draws_matrix.append(p_draw_matrix)
    p_draws_market.append(f["draw"])

p_draws_matrix = np.array(p_draws_matrix)
p_draws_market = np.array(p_draws_market)

print("=== P(draw) según MERCADO (Pinnacle fair) ===")
print(f"  Media:    {p_draws_market.mean():.3f}")
print(f"  Mediana:  {np.median(p_draws_market):.3f}")
print(f"  > 25%:    {(p_draws_market > 0.25).sum()} partidos")
print(f"  > 30%:    {(p_draws_market > 0.30).sum()} partidos")
print(f"  > 35%:    {(p_draws_market > 0.35).sum()} partidos")

print("\n=== P(draw) según MATRIZ POISSON ===")
print(f"  Media:    {p_draws_matrix.mean():.3f}")
print(f"  Mediana:  {np.median(p_draws_matrix):.3f}")
print(f"  > 25%:    {(p_draws_matrix > 0.25).sum()} partidos")
print(f"  > 30%:    {(p_draws_matrix > 0.30).sum()} partidos")
print(f"  > 35%:    {(p_draws_matrix > 0.35).sum()} partidos")

print("\n=== DIFERENCIA mercado vs matriz ===")
diff = p_draws_market - p_draws_matrix
print(f"  Media diferencia: {diff.mean():.3f}")
print(f"  El mercado tiene MAS P(draw) que Poisson: {(diff > 0).sum()} de 500 partidos")

KeyboardInterrupt: 

In [4]:
def predict_s1(row):
    """Poisson desde 1X2 solo — sin OU, prior neutro."""
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = {"over": 0.52, "under": 0.48}
    lh, la = infer_lambdas(f, fou)
    return build_matrix(lh, la)

def predict_s2(row):
    """Poisson desde 1X2 + OU."""
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = fair_ou(row["PC_over"], row["PC_under"])
    lh, la = infer_lambdas(f, fou)
    return build_matrix(lh, la)

def predict_s3(row):
    """Poisson desde 1X2 + OU + AH."""
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = fair_ou(row["PC_over"], row["PC_under"])
    lh, la = infer_lambdas(f, fou, ah_line=row["AHCh"])
    return build_matrix(lh, la)

def predict_s4(row):
    """Dixon-Coles sobre S3."""
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = fair_ou(row["PC_over"], row["PC_under"])
    lh, la = infer_lambdas(f, fou, ah_line=row["AHCh"])
    return apply_dixon_coles(build_matrix(lh, la), lh, la)

def predict_s5(row):
    """Baseline humano."""
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    p_home, p_away = f["home"], f["away"]
    fav_home = p_home >= p_away
    p_fav = max(p_home, p_away)
    if p_fav > 0.65:
        i, j = (2, 0) if fav_home else (0, 2)
    elif p_fav > 0.50:
        i, j = (1, 0) if fav_home else (0, 1)
    else:
        i, j = (1, 1)
    matrix = np.zeros((MAX_GOALS+1, MAX_GOALS+1))
    matrix[i, j] = 1.0
    return matrix

STRATEGIES = {
    #"S1_1x2":         predict_s1,
    "S2_1x2_ou":      predict_s2,
    "S3_1x2_ou_ah":   predict_s3,
    "S4_dixoncoles":  predict_s4,
    #"S5_baseline":    predict_s5,
}

SELECTORS = {
    #"maxExacto":    best_score_max_exact,
    "maxEV":        best_score_max_ev,
    #"maxEV_mkt25":  lambda m, p_draw_market=None: best_score_max_ev_with_draws(m, p_draw_market, 0.25),
    #"maxEV_mkt28":  lambda m, p_draw_market=None: best_score_max_ev_with_draws(m, p_draw_market, 0.28),
    #"maxEV_mkt30":  lambda m, p_draw_market=None: best_score_max_ev_with_draws(m, p_draw_market, 0.30),
}

In [37]:
# Pre-calcular puntos por partido para toda la data (correr una vez)
data["pts_s3_maxev"] = 0

for idx, row in tqdm(data.iterrows(), total=len(data)):
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = fair_ou(row["PC_over"], row["PC_under"])
    lh, la = infer_lambdas(f, fou, ah_line=row["AHCh"])
    matrix = build_matrix(lh, la)
    pred_h, pred_a = best_score_max_ev(matrix)
    data.at[idx, "pts_s3_maxev"] = score_points(pred_h, pred_a, int(row["FTHG"]), int(row["FTAG"]))

# Bootstrap sobre los puntos pre-calculados (instantáneo)
N_SIMULATIONS = 10000
puntos_simulados = [
    data["pts_s3_maxev"].sample(n=N_PARTIDOS, replace=True).sum()
    for _ in range(N_SIMULATIONS)
]
puntos_simulados = np.array(puntos_simulados)

print(f"=== BOOTSTRAP — {N_SIMULATIONS} simulaciones de {N_PARTIDOS} partidos ===")
print(f"  Media:          {puntos_simulados.mean():.1f} pts")
print(f"  Desviación std: {puntos_simulados.std():.1f} pts")
print(f"  Percentil 10:   {np.percentile(puntos_simulados, 10):.1f} pts")
print(f"  Percentil 25:   {np.percentile(puntos_simulados, 25):.1f} pts")
print(f"  Percentil 75:   {np.percentile(puntos_simulados, 75):.1f} pts")
print(f"  Percentil 90:   {np.percentile(puntos_simulados, 90):.1f} pts")

100%|██████████| 7988/7988 [13:46<00:00,  9.66it/s]  


=== BOOTSTRAP — 10000 simulaciones de 104 partidos ===
  Media:          204.7 pts
  Desviación std: 20.6 pts
  Percentil 10:   180.0 pts
  Percentil 25:   192.0 pts
  Percentil 75:   219.0 pts
  Percentil 90:   231.0 pts


In [11]:
N_SAMPLE = None # Cambiá a None para correr todos
data_run = data.sample(n=N_SAMPLE, random_state=42) if N_SAMPLE else data

results = {
    f"{s}_{sel}": {"points": 0, "exact": 0, "correct_result": 0, "n": 0,
                   "pred_home": 0, "pred_draw": 0, "pred_away": 0,
                   "predictions": []}
    for s in STRATEGIES
    for sel in SELECTORS
}

for _, row in tqdm(data_run.iterrows(), total=len(data_run)):
    real_h = int(row["FTHG"])
    real_a = int(row["FTAG"])

    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = fair_ou(row["PC_over"], row["PC_under"])

    lh_s2, la_s2 = infer_lambdas(f, fou)
    lh_s3, la_s3 = infer_lambdas(f, fou, ah_line=row["AHCh"])

    matrices = {}
    if "S2_1x2_ou" in STRATEGIES:
        matrices["S2_1x2_ou"] = build_matrix(lh_s2, la_s2)
    if "S3_1x2_ou_ah" in STRATEGIES:
        matrices["S3_1x2_ou_ah"] = build_matrix(lh_s3, la_s3)
    if "S4_dixoncoles" in STRATEGIES:
        matrices["S4_dixoncoles"] = apply_dixon_coles(build_matrix(lh_s3, la_s3), lh_s3, la_s3)

    for s_name, matrix in matrices.items():
        for sel_name, sel_fn in SELECTORS.items():
            key = f"{s_name}_{sel_name}"
            pred_h, pred_a = sel_fn(matrix)
            pts = score_points(pred_h, pred_a, real_h, real_a)
            r = results[key]
            r["points"] += pts
            r["n"] += 1

            if pred_h == real_h and pred_a == real_a:
                r["exact"] += 1
            elif get_result(pred_h, pred_a) == get_result(real_h, real_a):
                r["correct_result"] += 1

            res_pred = get_result(pred_h, pred_a)
            if res_pred == "home":   r["pred_home"] += 1
            elif res_pred == "draw": r["pred_draw"] += 1
            else:                    r["pred_away"] += 1

            if len(r["predictions"]) < 5:
                r["predictions"].append({
                    "partido": f"{row['HomeTeam']} vs {row['AwayTeam']}",
                    "pred": f"{pred_h}-{pred_a}",
                    "real": f"{real_h}-{real_a}",
                    "pts": pts,
                })

100%|██████████| 7988/7988 [15:08<00:00,  8.79it/s]


In [8]:
from collections import Counter

predicciones = []

for _, row in tqdm(data.iterrows(), total=len(data)):
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = fair_ou(row["PC_over"], row["PC_under"])
    lh, la = infer_lambdas(f, fou, ah_line=row["AHCh"])
    matrix = build_matrix(lh, la)
    pred_h, pred_a = best_score_max_ev(matrix)
    predicciones.append(f"{pred_h}-{pred_a}")

conteo = Counter(predicciones)
total = len(predicciones)

print("=== DISTRIBUCIÓN DE PREDICCIONES S3 maxEV ===\n")
for score, count in sorted(conteo.items(), key=lambda x: x[1], reverse=True):
    print(f"  {score}: {count} ({100*count/total:.1f}%)")

100%|██████████| 7988/7988 [07:34<00:00, 17.57it/s]


=== DISTRIBUCIÓN DE PREDICCIONES S3 maxEV ===

  1-0: 3156 (39.5%)
  0-1: 1785 (22.3%)
  2-1: 1018 (12.7%)
  2-0: 937 (11.7%)
  1-2: 745 (9.3%)
  0-2: 267 (3.3%)
  3-0: 62 (0.8%)
  0-0: 11 (0.1%)
  0-3: 6 (0.1%)
  3-1: 1 (0.0%)


In [12]:
print("=" * 80)
for key, r in sorted(results.items(), key=lambda x: x[1]["points"], reverse=True):
    n = r["n"]
    if n == 0: continue
    print(f"\n{'='*80}")
    print(f"  {key}")
    print(f"  Pts/partido: {r['points']/n:.3f} | Exacto%: {100*r['exact']/n:.1f}% | Resultado%: {100*(r['exact']+r['correct_result'])/n:.1f}%")
    print(f"  Predicciones → Local: {100*r['pred_home']/n:.1f}%  Empate: {100*r['pred_draw']/n:.1f}%  Visitante: {100*r['pred_away']/n:.1f}%")


  S2_1x2_ou_maxEV
  Pts/partido: 1.969 | Exacto%: 11.5% | Resultado%: 54.1%
  Predicciones → Local: 64.8%  Empate: 0.1%  Visitante: 35.1%

  S4_dixoncoles_maxEV
  Pts/partido: 1.967 | Exacto%: 11.8% | Resultado%: 53.7%
  Predicciones → Local: 65.8%  Empate: 3.4%  Visitante: 30.8%

  S3_1x2_ou_ah_maxEV
  Pts/partido: 1.963 | Exacto%: 11.5% | Resultado%: 53.9%
  Predicciones → Local: 68.3%  Empate: 0.0%  Visitante: 31.6%


In [ ]:
import json
import numpy as np
from scipy.optimize import minimize
from scipy.stats import poisson

# ── Cargar datos ─────────────────────────────────────────────────────────────
with open("C:/Users/Usuario/VS Code/Prediccion Prode/wc2026_markets.json") as f:
    matches = json.load(f)

print(f"Partidos cargados: {len(matches)}")

# ── S3 sobre datos del Mundial ────────────────────────────────────────────────
def matrix_ah_wc(matrix, line):
    i = np.arange(MAX_GOALS + 1)
    diff = i[:, None] - i[None, :]
    mask = diff + line > 0
    return float(np.sum(matrix[mask]))

def infer_lambdas_s3_wc(moneyline, total_main, spread_main):
    """S3 mejorado: 1X2 + OU principal + AH como restricción de probabilidad."""
    f1x2 = moneyline
    fou_line = total_main["line"]
    fou_over = total_main["over"]
    ah_line = spread_main["line"] if spread_main else None

    def loss(params):
        lh, la = params
        if lh <= 0 or la <= 0:
            return 1e6
        mat = build_matrix(lh, la)
        ph, pd_, pa = matrix_1x2(mat)
        i = np.arange(MAX_GOALS + 1)
        mask_ou = (i[:, None] + i[None, :]) > fou_line
        po = float(np.sum(mat[mask_ou]))
        err = ((ph - f1x2["home"])**2 +
               (pd_ - f1x2["draw"])**2 +
               (pa - f1x2["away"])**2 +
               (po - fou_over)**2)
        if ah_line is not None:
            p_ah = matrix_ah_wc(mat, ah_line)
            err += 0.3 * (p_ah - 0.5)**2
        return err

    share = f1x2["home"] / (f1x2["home"] + f1x2["away"] + 1e-9)
    total = max(1.5, min(2.5 + (fou_over - 0.5) * 2, 6.0))
    lh0 = total * (0.5 + 0.3 * (share - 0.5))
    la0 = total - lh0

    res = minimize(loss, x0=[lh0, la0], method="Nelder-Mead",
                   options={"xatol": 1e-5, "fatol": 1e-6, "maxiter": 2000})
    return max(res.x[0], 0.1), max(res.x[1], 0.1)

# ── Generar predicciones ──────────────────────────────────────────────────────
predictions_wc = []

for match in matches:
    m = match["markets"]
    try:
        lh, la = infer_lambdas_s3_wc(
            m["moneyline"],
            m["total_main"],
            m.get("spread_main")
        )
        mat = build_matrix(lh, la)
        pred_h, pred_a = best_score_max_ev(mat)

        # Top 5 scores más probables
        flat = [(i, j, float(mat[i, j]))
                for i in range(MAX_GOALS + 1)
                for j in range(MAX_GOALS + 1)]
        flat.sort(key=lambda x: -x[2])
        top5 = [f"{i}-{j} ({p*100:.1f}%)" for i, j, p in flat[:5]]

        predictions_wc.append({
            "home":       match["home"],
            "away":       match["away"],
            "start_time": match["start_time"][:10],
            "pred":       f"{pred_h}-{pred_a}",
            "lh":         round(lh, 2),
            "la":         round(la, 2),
            "top5":       top5,
        })
    except Exception as e:
        print(f"ERROR {match['home']} vs {match['away']}: {e}")

# ── Mostrar resultados ────────────────────────────────────────────────────────
predictions_wc.sort(key=lambda x: x["start_time"])

print(f"\n{'Fecha':<12} {'Local':<22} {'Visitante':<22} {'Pred':>6}  {'λh':>5} {'λa':>5}  Top 5")
print("-" * 110)
for p in predictions_wc:
    top = " | ".join(p["top5"])
    print(f"{p['start_time']:<12} {p['home']:<22} {p['away']:<22} {p['pred']:>6}  {p['lh']:>5} {p['la']:>5}  {top}")


match_test = next(m for m in matches if m["home"] == "Turkiye" and m["away"] == "USA")
print(json.dumps(match_test["markets"]["moneyline"], indent=2))

Partidos cargados: 72
ERROR Germany vs Curacao: 'NoneType' object is not subscriptable
ERROR Brazil vs Haiti: 'NoneType' object is not subscriptable

Fecha        Local                  Visitante                Pred     λh    λa  Top 5
--------------------------------------------------------------------------------------------------------------
2026-06-11   Mexico                 South Africa              2-0   2.03  0.66  2-0 (14.0%) | 1-0 (13.8%) | 3-0 (9.4%) | 2-1 (9.2%) | 1-1 (9.1%)
2026-06-12   South Korea            Czechia                   1-0   1.34  1.22  1-1 (12.6%) | 1-0 (10.4%) | 0-1 (9.4%) | 2-1 (8.5%) | 0-0 (7.7%)
2026-06-12   Canada                 Bosnia and Herzegovina    1-0   1.66  0.99  1-0 (11.7%) | 1-1 (11.6%) | 2-0 (9.7%) | 2-1 (9.6%) | 0-0 (7.1%)
2026-06-13   USA                    Paraguay                  1-0   1.52  1.01  1-1 (12.2%) | 1-0 (12.1%) | 2-1 (9.3%) | 2-0 (9.2%) | 0-1 (8.0%)
2026-06-13   Qatar                  Switzerland               0-2   0.49 

In [ ]:
# Comparar predicciones de maxEV_6_3 vs maxEV_3_1 en 20 partidos
muestra = data.sample(200, random_state=42)

diferencias = 0
for _, row in muestra.iterrows():
    f = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
    fou = fair_ou(row["PC_over"], row["PC_under"])
    lh, la = infer_lambdas(f, fou)
    matrix = build_matrix(lh, la)
    
    pred_63 = best_score_max_ev(matrix, pts_exacto=6, pts_resultado=3)
    pred_31 = best_score_max_ev(matrix, pts_exacto=3, pts_resultado=1)
    
    if pred_63 != pred_31:
        diferencias += 1
        print(f"{row['HomeTeam']} vs {row['AwayTeam']}")
        print(f"  maxEV_6_3: {pred_63}  |  maxEV_3_1: {pred_31}")
        f1x2 = fair_1x2(row["PSCH"], row["PSCD"], row["PSCA"])
        print(f"  P(home)={f1x2['home']:.2f}  P(draw)={f1x2['draw']:.2f}  P(away)={f1x2['away']:.2f}")

print(f"\nDiferencias: {diferencias} de 200 partidos")


Diferencias: 0 de 20 partidos
